## Imports & configuration

In [20]:
import pandas as pd
import numpy as np
import requests
import time
import os
import math
import psycopg2
from psycopg2.extras import execute_values
import boto3
from dotenv import load_dotenv
from datetime import date
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from parsel import Selector
import plotly.express as px
import plotly.graph_objects as go
from urllib.parse import quote_plus

load_dotenv(dotenv_path='../.env')
print('Librairies chargées ✓')

Librairies chargées ✓


## Les 35 villes cibles

In [21]:
VILLES = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen",
    "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon", "Annecy", "Grenoble", "Lyon",
    "Gorges du Verdon", "Bormes les Mimosas", "Cassis", "Marseille",
    "Aix en Provence", "Avignon", "Uzes", "Nimes", "Aigues Mortes",
    "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege",
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle"
]
print(f"{len(VILLES)} villes chargées")

35 villes chargées


## Étape 1 — Météo (Nominatim + OpenWeatherMap)

Pour chaque ville :
1. Coordonnées GPS via Nominatim
2. Prévisions 7 jours via One Call API 3.0
3. Calcul du `beauty_score` = moyenne(score_pluie, score_temp, score_humidité)


In [22]:
api_key = os.getenv('api_key')

data_frames = []

for id_ville, ville in enumerate(VILLES, start=1):
    # Géolocalisation
    params_geo = {
        'q': ville, 'format': 'json', 'addressdetails': 1,
        'limit': 1, 'email': 'julien.charlier31@gmail.com'
    }
    r_geo = requests.get(
        'https://nominatim.openstreetmap.org/search',
        params=params_geo, headers={'User-Agent': 'MonApp/1.0'}
    )
    if r_geo.status_code != 200 or not r_geo.json():
        print(f'Géoloc KO : {ville}')
        time.sleep(1)
        continue

    geo = r_geo.json()[0]
    lat, lon = float(geo['lat']), float(geo['lon'])
    country = geo.get('address', {}).get('country', 'Unknown')

    # Météo 7 jours
    params_w = {
        'lat': lat, 'lon': lon, 'appid': api_key,
        'units': 'metric', 'exclude': 'minutely,hourly,alerts'
    }
    r_w = requests.get('https://api.openweathermap.org/data/3.0/onecall', params=params_w)
    if r_w.status_code != 200:
        print(f'Météo KO : {ville} ({r_w.status_code})')
        time.sleep(1)
        continue

    daily = r_w.json().get('daily', [])[:7]
    if not daily:
        print(f'Pas de données daily : {ville}')
        continue

    fetch_date       = date.today().isoformat()
    forecast_end     = date.fromtimestamp(daily[-1]['dt']).isoformat()

    avg_temp     = sum(d['temp']['day'] for d in daily) / len(daily)
    avg_humidity = sum(d['humidity']    for d in daily) / len(daily)
    total_rain   = sum(d.get('rain', 0) for d in daily)
    avg_pop      = sum(d.get('pop', 0)  for d in daily) / len(daily)

    rain_score     = max(0, 100 - total_rain * 10)
    temp_score     = max(0, 100 - abs(avg_temp - 20) * 5)
    humidity_score = max(0, 100 - (avg_humidity - 40) * 2)
    beauty_score   = round((rain_score + temp_score + humidity_score) / 3, 2)

    data_frames.append(pd.DataFrame([{
        'id': id_ville, 'city': ville,
        'latitude': lat, 'longitude': lon, 'country': country,
        'fetch_date': fetch_date,
        'forecast_end_date': forecast_end,
        'avg_temp_7j': round(avg_temp, 2),
        'avg_humidity_7j': round(avg_humidity, 2),
        'total_rain_7j': round(total_rain, 2),
        'avg_pop_7j': round(avg_pop * 100, 2),
        'beauty_score': beauty_score
    }]))

    print(f'✓ {ville:<35} score: {beauty_score:.2f}  [{fetch_date} → {forecast_end}]')
    time.sleep(1.5)

if not data_frames:
    raise RuntimeError('Aucune donnée météo récupérée. Vérifiez la clé API OpenWeather.')

df_weather = (
    pd.concat(data_frames, ignore_index=True)
    .sort_values('beauty_score', ascending=False)
    .reset_index(drop=True)
)
df_weather.to_csv('../data/processed/villes_meteo.csv', index=False, encoding='utf-8')
print(f'\n✓ {len(df_weather)} villes — sauvegardé dans data/processed/villes_meteo.csv')

✓ Mont Saint Michel                   score: 45.80  [2026-05-28 → 2026-06-03]
✓ St Malo                             score: 42.58  [2026-05-28 → 2026-06-03]
✓ Bayeux                              score: 45.01  [2026-05-28 → 2026-06-03]
✓ Le Havre                            score: 40.73  [2026-05-28 → 2026-06-03]
✓ Rouen                               score: 47.93  [2026-05-28 → 2026-06-03]
✓ Paris                               score: 56.57  [2026-05-28 → 2026-06-03]
✓ Amiens                              score: 49.08  [2026-05-28 → 2026-06-03]
✓ Lille                               score: 52.60  [2026-05-28 → 2026-06-03]
✓ Strasbourg                          score: 52.04  [2026-05-28 → 2026-06-03]
✓ Chateau du Haut Koenigsbourg        score: 57.19  [2026-05-28 → 2026-06-03]
✓ Colmar                              score: 58.85  [2026-05-28 → 2026-06-03]
✓ Eguisheim                           score: 63.75  [2026-05-28 → 2026-06-03]
✓ Besancon                            score: 50.59  [2026-05-28 

In [23]:
df_weather.head(10)

,id,city,latitude,longitude,country,fetch_date,forecast_end_date,avg_temp_7j,avg_humidity_7j,total_rain_7j,avg_pop_7j,beauty_score
0,22,Aix en Provence,43.529842,5.447474,France,2026-05-28,2026-06-03,28.90,35.86,1.43,21.57,83.16
1,25,Nimes,43.837425,4.360069,France,2026-05-28,2026-06-03,30.00,31.71,1.92,28.57,82.45
2,24,Uzes,44.012128,4.419672,France,2026-05-28,2026-06-03,28.53,36.29,2.63,35.57,79.50
3,21,Marseille,43.296399,5.377789,France,2026-05-28,2026-06-03,25.48,56.86,1.04,17.57,76.17
4,20,Cassis,43.214036,5.539632,France,2026-05-28,2026-06-03,23.39,61.57,1.16,17.71,76.10
5,19,Bormes les Mimosas,43.150697,6.341928,France,2026-05-28,2026-06-03,23.47,64.29,0.77,14.86,75.47
6,31,Toulouse,43.604464,1.444243,France,2026-05-28,2026-06-03,25.12,48.29,3.75,14.29,73.45
7,26,Aigues Mortes,43.566152,4.191540,France,2026-05-28,2026-06-03,26.52,48.43,3.03,28.57,73.41
8,29,Carcassonne,43.213036,2.349107,France,2026-05-28,2026-06-03,26.74,42.43,4.23,14.29,73.04
9,27,Saintes Maries de la mer,43.451592,4.427720,France,2026-05-28,2026-06-03,24.76,57.57,3.03,15.57,70.25


## Étape 2 — Scraping Booking.com (Selenium)

Pour chaque ville, extraction du **top 3 hôtels** (par note utilisateurs) et géocodage des coordonnées GPS via Nominatim.


In [ ]:
def scrape_booking_ville(ville, n_hotels=3):
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    options.add_argument(
        '--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    )
    driver = webdriver.Chrome(options=options)
    url = (
        'https://www.booking.com/searchresults.html'
        f'?ss={quote_plus(ville)}&lang=fr&group_adults=2&no_rooms=1'
        '&group_children=0&order=review_score_and_count'
    )
    selector = None
    try:
        driver.get(url)
        time.sleep(3)
        for _ in range(3):
            driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
            time.sleep(2)
        selector = Selector(text=driver.page_source)
    except Exception as e:
        print(f'  Erreur Selenium pour {ville}: {e}')
        return pd.DataFrame()
    finally:
        driver.quit()

    if selector is None:
        return pd.DataFrame()

    hotels_blocks = selector.css("div[class='aa97d6032f']")
    data = []
    for block in hotels_blocks:
        name        = block.css("div[class='b87c397a13 a3e0b4ffd1']::text").get()
        note        = block.css("div[class='f63b14ab7a dff2e52086']::text").get()
        description = block.css("div[class='fff1944c52']::text").get()
        link        = block.css("a[class='bd77474a8e']::attr(href)").get()
        if name and note:
            try:
                note_float = float(note.replace(',', '.'))
            except ValueError:
                continue
            data.append({
                'city': ville, 'hotel_name': name.strip(),
                'note': note_float,
                'description': description.strip() if description else 'N/A',
                'lien': link
            })

    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    return df.sort_values('note', ascending=False).head(n_hotels).reset_index(drop=True)


def geocode_hotel(hotel_name, city):
    params = {
        'q': f'{hotel_name}, {city}, France',
        'format': 'json', 'limit': 1,
        'email': 'julien.charlier31@gmail.com'
    }
    try:
        r = requests.get(
            'https://nominatim.openstreetmap.org/search',
            params=params, headers={'User-Agent': 'MonApp/1.0'}, timeout=5
        )
        if r.status_code == 200 and r.json():
            result = r.json()[0]
            return float(result['lat']), float(result['lon'])
    except Exception:
        pass
    return None, None


all_hotels = []

for ville in VILLES:
    print(f'Scraping {ville}...')
    try:
        df_ville = scrape_booking_ville(ville)
    except Exception as e:
        print(f'  Erreur inattendue pour {ville}: {e}')
        continue

    if df_ville.empty:
        print(f'  Aucun résultat pour {ville}')
        time.sleep(2)
        continue

    city_row = df_weather[df_weather['city'] == ville]
    city_lat = city_row.iloc[0]['latitude']  if not city_row.empty else None
    city_lon = city_row.iloc[0]['longitude'] if not city_row.empty else None

    lats, lons = [], []
    for _, row in df_ville.iterrows():
        lat, lon = geocode_hotel(row['hotel_name'], ville)
        lats.append(lat if lat is not None else city_lat)
        lons.append(lon if lon is not None else city_lon)
        time.sleep(1.2)

    df_ville['latitude']  = lats
    df_ville['longitude'] = lons
    all_hotels.append(df_ville)
    print(f'  ✓ {len(df_ville)} hôtels')

if not all_hotels:
    raise RuntimeError('Aucun hôtel scrappé. Vérifiez les sélecteurs CSS Booking.com.')

df_hotels = pd.concat(all_hotels, ignore_index=True)
df_hotels.insert(0, 'id', range(1, len(df_hotels) + 1))
df_hotels.to_csv('../data/processed/df_top3_hotels.csv', index=False, encoding='utf-8')
print(f'\n✓ {len(df_hotels)} hôtels — sauvegardé dans data/processed/df_top3_hotels.csv')

In [62]:
df_hotels.head(10)

,id,city,hotel_name,note,description,lien,latitude,longitude
0,1,Mont Saint Michel,La Villa du Manoir,10.0,"Situé à Beauvoir, l’hébergement La Villa du Ma...",https://www.booking.com/hotel/fr/la-villa-du-m...,48.635954,-1.511460
1,2,Mont Saint Michel,Les mouettes,9.9,Hébergement géré par un particulier,https://www.booking.com/hotel/fr/les-mouettes-...,48.635422,-1.510084
2,3,Mont Saint Michel,le coin des hirondelles,9.8,Hébergement géré par un particulier,https://www.booking.com/hotel/fr/le-coin-des-h...,48.635954,-1.511460
3,4,St Malo,Appartement de charme,9.8,Hébergement géré par un particulier,https://www.booking.com/hotel/fr/appartement-d...,48.649518,-2.026041
4,5,St Malo,Superbe Apt Saint-Malo With View,9.8,Hébergement géré par un particulier,https://www.booking.com/hotel/fr/superbe-apt-s...,48.649518,-2.026041
5,6,St Malo,Le Pilo,9.8,Offrant une vue sur la ville et doté d’une con...,https://www.booking.com/hotel/fr/le-pilori.fr....,48.649518,-2.026041
6,7,Bayeux,Le Chat Qui Veille,10.0,L’hébergement Le Chat Qui Veille vous accueill...,https://www.booking.com/hotel/fr/gite-6-47-9-p...,49.276462,-0.702474
7,8,Bayeux,Historic 18th-Century Mansion - 3 BR - Bayeux ...,9.9,Hébergement géré par un particulier,https://www.booking.com/hotel/fr/bayeux-norman...,49.276462,-0.702474
8,9,Bayeux,"Hôtel particulier ""le clos de la croix""",9.8,"Situé à Bayeux, l’établissement Hôtel particul...",https://www.booking.com/hotel/fr/particulier-q...,49.276462,-0.702474
9,10,Le Havre,Vivez au coeur Historique - St François - Gran...,9.5,Hébergement géré par un particulier,https://www.booking.com/hotel/fr/nouveau-vivez...,49.493898,0.107973


## Étape 3 — Data Lake (AWS S3)

Upload des deux datasets nettoyés dans le bucket S3.


In [63]:
BUCKET_NAME = 'dsfs-ft-38-charlier-julien'

s3 = boto3.resource(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name='eu-west-3'
)
bucket = s3.Bucket(BUCKET_NAME)

bucket.upload_file('../data/processed/villes_meteo.csv',    'villes_meteo.csv')
bucket.upload_file('../data/processed/df_top3_hotels.csv',  'df_top3_hotels.csv')

print(f'✓ villes_meteo.csv    → s3://{BUCKET_NAME}/')
print(f'✓ df_top3_hotels.csv  → s3://{BUCKET_NAME}/')

✓ villes_meteo.csv    → s3://dsfs-ft-38-charlier-julien/
✓ df_top3_hotels.csv  → s3://dsfs-ft-38-charlier-julien/


## Étape 4 — ETL (S3 → Neon DB)

**Extract** : lecture des fichiers depuis S3  
**Transform** : vérification des types  
**Load** : insertion dans Neon DB (PostgreSQL)


### 4a — Extract depuis S3

In [64]:
s3_client = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name='eu-west-3'
)

obj_weather = s3_client.get_object(Bucket=BUCKET_NAME, Key='villes_meteo.csv')
df_etl_weather = pd.read_csv(obj_weather['Body'])

obj_hotels = s3_client.get_object(Bucket=BUCKET_NAME, Key='df_top3_hotels.csv')
df_etl_hotels = pd.read_csv(obj_hotels['Body'])

print(f'Extrait depuis S3 : {len(df_etl_weather)} villes, {len(df_etl_hotels)} hôtels')

Extrait depuis S3 : 35 villes, 105 hôtels


### 4a.5 — Transform

In [ ]:
from datetime import date, timedelta

if 'fetch_date' not in df_etl_weather.columns:
    df_etl_weather['fetch_date'] = date.today().isoformat()
    df_etl_weather['forecast_end_date'] = (date.today() + timedelta(days=7)).isoformat()

print("=" * 55)
print("TRANSFORM — Nettoyage & validation")
print("=" * 55)

dup_w = df_etl_weather.duplicated(subset=['city']).sum()
dup_h = df_etl_hotels.duplicated(subset=['city', 'hotel_name']).sum()
df_etl_weather = df_etl_weather.drop_duplicates(subset=['city'])
df_etl_hotels  = df_etl_hotels.drop_duplicates(subset=['city', 'hotel_name'])
print(f"\n[1] Doublons supprimés — weather: {dup_w}  |  hotels: {dup_h}")

invalid_score = ~df_etl_weather['beauty_score'].between(0, 100)
invalid_note  = ~df_etl_hotels['note'].between(0, 10)

if invalid_score.any():
    print(f"\n[2] beauty_score hors [0-100] → clamp :")
    print(df_etl_weather[invalid_score][['city', 'beauty_score']])
    df_etl_weather['beauty_score'] = df_etl_weather['beauty_score'].clip(0, 100)
else:
    print("\n[2] beauty_score : toutes les valeurs dans [0-100] ✓")

if invalid_note.any():
    print(f"    note hors [0-10] → clamp :")
    print(df_etl_hotels[invalid_note][['city', 'hotel_name', 'note']])
    df_etl_hotels['note'] = df_etl_hotels['note'].clip(0, 10)
else:
    print("    note hôtel    : toutes les valeurs dans [0-10]  ✓")

df_etl_hotels['hotel_name']  = df_etl_hotels['hotel_name'].str.strip()
df_etl_hotels['description'] = (
    df_etl_hotels['description']
    .fillna('N/A')
    .str.strip()
    .replace('', 'N/A')
)
df_etl_weather['city']    = df_etl_weather['city'].str.strip()
df_etl_weather['country'] = df_etl_weather['country'].str.strip()
print("\n[3] Normalisation des chaînes (strip, fillna) ✓")

LAT_MIN, LAT_MAX = 41.0, 51.5
LON_MIN, LON_MAX =  -5.5, 10.0

bad_w = ~(
    df_etl_weather['latitude'].between(LAT_MIN, LAT_MAX) &
    df_etl_weather['longitude'].between(LON_MIN, LON_MAX)
)
bad_h = ~(
    df_etl_hotels['latitude'].between(LAT_MIN, LAT_MAX) &
    df_etl_hotels['longitude'].between(LON_MIN, LON_MAX)
)

if bad_w.any():
    print(f"\n[4] Coordonnées weather incohérentes ({bad_w.sum()} lignes) :")
    print(df_etl_weather[bad_w][['city', 'latitude', 'longitude']])
else:
    print("\n[4] Coordonnées weather : toutes dans les bornes France ✓")

if bad_h.any():
    print(f"    Coordonnées hôtels incohérentes ({bad_h.sum()} lignes) :")
    print(df_etl_hotels[bad_h][['city', 'hotel_name', 'latitude', 'longitude']])
else:
    print("    Coordonnées hôtels  : toutes dans les bornes France ✓")

before = len(df_etl_hotels)
df_etl_hotels = df_etl_hotels.dropna(subset=['latitude', 'longitude'])
removed = before - len(df_etl_hotels)
print(f"\n[5] Hôtels sans coordonnées supprimés : {removed}")

print(f"\n{'=' * 55}")
print(f"Après nettoyage : {len(df_etl_weather)} villes | {len(df_etl_hotels)} hôtels")
print(f"Période : {df_etl_weather['fetch_date'].iloc[0]} → {df_etl_weather['forecast_end_date'].iloc[0]}")
print(f"{'=' * 55}")

TRANSFORM — Nettoyage & validation

[1] Doublons supprimés — weather: 0  |  hotels: 0

[2] beauty_score : toutes les valeurs dans [0-100] ✓
    note hôtel    : toutes les valeurs dans [0-10]  ✓

[3] Normalisation des chaînes (strip, fillna) ✓

[4] Coordonnées weather : toutes dans les bornes France ✓
    Coordonnées hôtels  : toutes dans les bornes France ✓

[5] Hôtels sans coordonnées supprimés : 0

Après nettoyage : 35 villes | 105 hôtels
Période : 2026-05-28 → 2026-06-03


### 4b — Création des tables (si nécessaire)

In [66]:
def get_pg_conn():
    return psycopg2.connect(
        host=os.getenv('PGHOST'),
        dbname=os.getenv('PGDATABASE'),
        user=os.getenv('PGUSER'),
        password=os.getenv('PGPASSWORD'),
        sslmode=os.getenv('PGSSLMODE', 'require')
    )

conn = get_pg_conn()
cur  = conn.cursor()

cur.execute("DROP TABLE IF EXISTS weather;")
cur.execute("""
    CREATE TABLE weather (
        id                INTEGER PRIMARY KEY,
        city              VARCHAR(100),
        latitude          FLOAT,
        longitude         FLOAT,
        country           VARCHAR(100),
        fetch_date        DATE,
        forecast_end_date DATE,
        avg_temp_7j       FLOAT,
        avg_humidity_7j   FLOAT,
        total_rain_7j     FLOAT,
        avg_pop_7j        FLOAT,
        beauty_score      FLOAT
    );
""")

cur.execute("""
    CREATE TABLE IF NOT EXISTS hotels (
        id          INTEGER PRIMARY KEY,
        city        VARCHAR(100),
        hotel_name  VARCHAR(255),
        note        FLOAT,
        description TEXT,
        lien        TEXT,
        latitude    FLOAT,
        longitude   FLOAT
    );
""")

conn.commit()
cur.close()
conn.close()
print('✓ Tables weather et hotels créées')

✓ Tables weather et hotels créées


### 4c — Load dans Neon DB

In [ ]:
def _clean(v):
    """Convertit NaN en None pour psycopg2."""
    if v is None:
        return None
    try:
        if math.isnan(float(v)):
            return None
    except (TypeError, ValueError):
        pass
    return v


conn = get_pg_conn()
cur  = conn.cursor()

cur.execute('DELETE FROM hotels;')

weather_rows = [
    (
        int(r.id), r.city, r.latitude, r.longitude, r.country,
        r.fetch_date, r.forecast_end_date,
        r.avg_temp_7j, r.avg_humidity_7j, r.total_rain_7j,
        r.avg_pop_7j, r.beauty_score
    )
    for r in df_etl_weather.itertuples()
]
execute_values(cur, """
    INSERT INTO weather
        (id, city, latitude, longitude, country,
         fetch_date, forecast_end_date,
         avg_temp_7j, avg_humidity_7j, total_rain_7j, avg_pop_7j, beauty_score)
    VALUES %s;
""", weather_rows)

hotels_rows = [
    (
        int(r.id), r.city, r.hotel_name, _clean(r.note),
        _clean(r.description), _clean(r.lien),
        _clean(r.latitude), _clean(r.longitude)
    )
    for r in df_etl_hotels.itertuples()
]
execute_values(cur, """
    INSERT INTO hotels (id, city, hotel_name, note, description, lien, latitude, longitude)
    VALUES %s;
""", hotels_rows)

conn.commit()
cur.close()
conn.close()
print(f'✓ {len(weather_rows)} villes et {len(hotels_rows)} hôtels chargés dans Neon DB')

✓ 35 villes et 105 hôtels chargés dans Neon DB


### 4d — Vérification de la base

In [68]:
conn = get_pg_conn()
cur  = conn.cursor()

cur.execute('SELECT COUNT(*) FROM weather;')
n_w = cur.fetchone()[0]

cur.execute('SELECT COUNT(*) FROM hotels;')
n_h = cur.fetchone()[0]

cur.execute(
    'SELECT city, beauty_score, fetch_date, forecast_end_date FROM weather ORDER BY beauty_score DESC LIMIT 5;'
)
top5_db = cur.fetchall()

cur.close()
conn.close()

print(f'weather : {n_w} lignes')
print(f'hotels  : {n_h} lignes')
print('\nTop 5 destinations en DB :')
for city, score, fd, fend in top5_db:
    print(f'  {city:<30} {score:.2f}   [{fd} → {fend}]')

weather : 35 lignes
hotels  : 105 lignes

Top 5 destinations en DB :
  Aix en Provence                83.16   [2026-05-28 → 2026-06-03]
  Nimes                          82.45   [2026-05-28 → 2026-06-03]
  Uzes                           79.50   [2026-05-28 → 2026-06-03]
  Marseille                      76.17   [2026-05-28 → 2026-06-03]
  Cassis                         76.10   [2026-05-28 → 2026-06-03]


## Étape 5 — Data Visualisation

- **Carte 1** : Beauty score des 35 villes + top 3 hôtels au survol
- **Carte 2** : Température moyenne 7 jours + top 3 hôtels au survol
- **Carte 3** : Localisation et notes Booking des 105 hôtels

In [ ]:
df_weather = pd.read_csv('../data/processed/villes_meteo.csv')
df_hotels  = pd.read_csv('../data/processed/df_top3_hotels.csv')

def get_hotels_text(city):
    hotels = df_hotels[df_hotels['city'] == city]
    if hotels.empty:
        return "Aucun hôtel trouvé"
    text = "<b>Top 3 Hôtels :</b><br>"
    for i, (_, row) in enumerate(hotels.iterrows(), 1):
        text += f"{i}. <b>{row['hotel_name']}</b> — {row['note']}/10<br>"
    return text

df_weather['hotels_text'] = df_weather['city'].apply(get_hotels_text)

score_min, score_max = df_weather['beauty_score'].min(), df_weather['beauty_score'].max()
df_weather['size_score'] = 5 + (df_weather['beauty_score'] - score_min) / (score_max - score_min) * 20

temp_min, temp_max = df_weather['avg_temp_7j'].min(), df_weather['avg_temp_7j'].max()
df_weather['size_temp'] = 5 + (df_weather['avg_temp_7j'] - temp_min) / (temp_max - temp_min) * 20

print(f'{len(df_weather)} villes, {len(df_hotels)} hôtels chargés')

35 villes, 105 hôtels chargés


### Carte 1 — Beauty Score des 35 villes

In [70]:
fetch = df_weather['fetch_date'].iloc[0]
end   = df_weather['forecast_end_date'].iloc[0]

fig_beauty = px.scatter_map(
    df_weather,
    lat='latitude',
    lon='longitude',
    hover_name='city',
    color='beauty_score',
    size='size_score',
    color_continuous_scale='RdYlGn',
    size_max=18,
    zoom=4,
    map_style='carto-positron',
    hover_data={'latitude': False, 'longitude': False, 'size_score': False}
)
fig_beauty.update_traces(
    customdata=df_weather['hotels_text'],
    hovertemplate='<b>%{hovertext}</b><br>Beauty Score : %{marker.color:.1f}<br>%{customdata}<extra></extra>'
)
fig_beauty.update_layout(
    title=f'Beauty Score météo ({fetch} → {end})',
    height=650,
    width=550,
    margin=dict(l=0, r=0, b=0, t=50),
    coloraxis_colorbar=dict(title='Score')
)
fig_beauty.show()

### Carte 2 — Température moyenne 7 jours des 35 villes

In [71]:
fig_temp = px.scatter_map(
    df_weather,
    lat='latitude',
    lon='longitude',
    hover_name='city',
    color='avg_temp_7j',
    size='size_temp',
    color_continuous_scale='Turbo',
    size_max=18,
    zoom=4,
    map_style='carto-positron',
    hover_data={'latitude': False, 'longitude': False, 'size_temp': False}
)
fig_temp.update_traces(
    customdata=df_weather['hotels_text'],
    hovertemplate='<b>%{hovertext}</b><br>Temp. moy. : %{marker.color:.1f}°C<br>%{customdata}<extra></extra>'
)
fig_temp.update_layout(
    title=f'Température moyenne 7 jours — 35 villes françaises ({fetch} → {end})',
    height=650,
    width=550,
    margin=dict(l=0, r=0, b=0, t=50),
    coloraxis_colorbar=dict(title='°C')
)
fig_temp.show()